In [ ]:
!pip install -q datasets soundfile transformers accelerate torch resemblyzer librosa pandas scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from pathlib import Path
from scipy.signal import resample
from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav

# Updated Config
NUM_SAMPLES = 30
MODEL_NAME = "facebook/mms-tts-urd-script_arabic"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading models to {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
encoder = VoiceEncoder()

def run_evaluation(dataset_name, samples_list, text_key):
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated", "anchor"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i in range(min(NUM_SAMPLES, len(samples_list))):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"
            anc_path = f"{base_dir}/anchor/sample_{i}.wav"

            # 1. Save Reference
            sf.write(ref_path, ref_array, ref_sr)

            # 2. Generate TTS
            inputs = tokenizer(text, return_tensors="pt").to(device)
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # 3. Create Anchor (8kHz resample cycle)
            target_len_8k = int(len(ref_array) * 8000 / ref_sr)
            downsampled = resample(ref_array, target_len_8k)
            upsampled = resample(downsampled, len(ref_array))
            sf.write(anc_path, upsampled, ref_sr)

            # 4. Metrics
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))

            emb_ref_full = encoder.embed_utterance(wav_ref)
            emb_gen = encoder.embed_utterance(wav_gen)

            sim = np.dot(emb_ref_full, emb_gen) / (np.linalg.norm(emb_ref_full) * np.linalg.norm(emb_gen))

            results.append({
                "sample_id": i,
                "urdu_text": text,
                "ref_gen_similarity": float(sim)
            })
            print(f"[{dataset_name.upper()}] Sample {i+1}/{NUM_SAMPLES}: Similarity={sim:.4f}")

        except Exception as e:
            print(f"Error in {dataset_name} sample {i}: {e}")
            continue

    return results

Loading models to cuda...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loaded the voice encoder model on cuda in 0.12 seconds.


In [ ]:
from datasets import load_dataset

print("🔄 Loading Fleurs (ur_pk) test split in streaming mode...")

# Fetch Fleurs stream - this is the exact recommended way from the official dataset card
ds_fleurs_stream = load_dataset(
    "google/fleurs",
    "ur_pk",
    split="test",
    trust_remote_code=True,
    streaming=True
)

# Collect exactly 30 samples
fleurs_samples = []
for i, sample in enumerate(ds_fleurs_stream):
    fleurs_samples.append(sample)
    if len(fleurs_samples) >= NUM_SAMPLES:
        break

print(f"✅ Successfully loaded {len(fleurs_samples)} samples from Fleurs ur_pk test split")
if len(fleurs_samples) > 0:
    print("   Sample keys:", list(fleurs_samples[0].keys()))
    print("   First transcription:", fleurs_samples[0]["transcription"][:100] + "...")

# Execute the evaluation function (your original function from Step 2)
results_fleurs = run_evaluation("fleurs", fleurs_samples, text_key="transcription")

# Save stats to CSV
df_fleurs = pd.DataFrame(results_fleurs)
df_fleurs.to_csv("fleurs_evaluation_results.csv", index=False)

print("\n🎉 Fleurs Evaluation Complete!")
print(f"   Average Similarity: {df_fleurs['ref_gen_similarity'].mean():.4f}")
print(f"   Results saved to fleurs_evaluation_results.csv")

🔄 Loading Fleurs (ur_pk) test split in streaming mode...
✅ Successfully loaded 30 samples from Fleurs ur_pk test split
   Sample keys: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']
   First transcription: ان کے دور میں خفیف تلوُّث ایزا مسئلہ نہیں تھا جیسا کہ وہ آج ہے ان کی سکونت ایسے شہروں یا حوزات میں ہ...
[FLEURS] Sample 1/30: Similarity=0.5287
[FLEURS] Sample 2/30: Similarity=0.5672
[FLEURS] Sample 3/30: Similarity=0.5599
[FLEURS] Sample 4/30: Similarity=0.5346
[FLEURS] Sample 5/30: Similarity=0.5309
[FLEURS] Sample 6/30: Similarity=0.5069
[FLEURS] Sample 7/30: Similarity=0.5894
[FLEURS] Sample 8/30: Similarity=0.5367
[FLEURS] Sample 9/30: Similarity=0.5376
[FLEURS] Sample 10/30: Similarity=0.5103
[FLEURS] Sample 11/30: Similarity=0.5140
[FLEURS] Sample 12/30: Similarity=0.5383
[FLEURS] Sample 13/30: Similarity=0.5059
[FLEURS] Sample 14/30: Similarity=0.5327
[FLEURS] Sample 15/30: Similarity=0.5645
[FLE

In [ ]:
import zipfile
import os
from google.colab import files

# Define the folder created by the evaluation function
folder_to_zip = "mushra_fleurs"
zip_name = "mushra_fleurs_results.zip"

with zipfile.ZipFile(zip_name, "w") as zf:
    for root, dirs, files_in_dir in os.walk(folder_to_zip):
        for file in files_in_dir:
            zf.write(os.path.join(root, file))

print(f"Created {zip_name}")

# Download both the ZIP and the CSV
# Fixed: using the correct filename generated in the previous step
files.download(zip_name)
files.download("fleurs_evaluation_results.csv")

Created mushra_fleurs_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>